# Data Exploration

Exhaustive look at every credit-risk dataset used in this benchmark.
Each section is a thin caller of `src.visualizations.data_exploration`
so the analysis stays out of the notebook and can be unit-tested.

Sections:
1. **Setup** -- paths, imports, figure cleanup
2. **Raw data inventory** -- file sizes, row counts, dtype counts (per task)
3. **Processed data inventory** -- after `src/data/preprocessing.py` ran
4. **PD: class balance** -- positive rate, imbalance ratio per dataset
5. **LGD: target distribution** -- mean / skew / % zeros, plus histograms
6. **Missingness** -- per-dataset and aggregated tables + heatmaps
7. **Numeric feature stats** -- per-feature descriptive statistics
8. **Feature correlations** -- per-dataset correlation heatmaps
9. **Geometry: 2D PCA scatter** -- coloured by target

All figures land under `figures/data_exploration/` (cleared at notebook
start so re-runs give a clean output set).


## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.visualizations.experiment_plots import apply_style, reset_figure_dir
from src.visualizations.data_exploration import (
    list_raw_datasets, list_processed_datasets,
    raw_dataset_summary_table, processed_dataset_summary_table,
    pd_target_balance_table, lgd_target_distribution_table,
    missingness_table, numeric_feature_stats,
    plot_dataset_size_bar, plot_target_balance, plot_lgd_target_hists,
    plot_missingness_heatmap, plot_correlation_heatmap, plot_pca_2d,
)
apply_style()

FIGURES_DIR = PROJECT_ROOT / 'figures' / 'data_exploration'
FIGURES_DIR = reset_figure_dir(FIGURES_DIR)  # wipe previous figures
print(f'Figures will be saved to: {FIGURES_DIR}')
print(f'Raw  PD  datasets: {len(list_raw_datasets("pd"))}')
print(f'Raw  LGD datasets: {len(list_raw_datasets("lgd"))}')
print(f'Proc PD  datasets: {len(list_processed_datasets("pd"))}')
print(f'Proc LGD datasets: {len(list_processed_datasets("lgd"))}')


Figures will be saved to: c:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\figures\data_exploration
Raw  PD  datasets: 13
Raw  LGD datasets: 7
Proc PD  datasets: 14
Proc LGD datasets: 7


## 2. Raw data inventory

Walks every CSV under `data/raw/{pd,lgd}/` and reports row count, column
count, file size, and a dtype breakdown. The `columns_preview` field shows
the first few column names so you can spot potential ID / date / target
columns at a glance.


In [2]:
raw_pd = raw_dataset_summary_table('pd')
print('RAW PD overview:')
display(raw_pd.sort_values('rows', ascending=False))
print(f'Total rows across all PD datasets: {int(raw_pd["rows"].sum()):,}')


RAW PD overview:


,dataset,task,rows,cols,size_MB,n_numeric_cols,n_object_cols,columns_preview
5,0006.hackerearth,pd,532428,45,137.47,28,17,"member_id, loan_amnt, funded_amnt, funded_amnt..."
11,0012.home_credit,pd,307511,122,158.44,106,16,"SK_ID_CURR, TARGET, NAME_CONTRACT_TYPE, CODE_G..."
2,0003.vehicle_loan,pd,233154,41,39.89,35,6,"UniqueID, disbursed_amount, asset_cost, ltv, b..."
0,0001.gmsc,pd,150000,11,6.32,11,0,"SeriousDlqin2yrs, RevolvingUtilizationOfUnsecu..."
10,0011.loan_default,pd,105471,771,487.29,759,12,"id, f1, f2, f3, f4, f5..."
8,0009.bank_status,pd,100514,19,17.94,12,7,"Loan ID, Customer ID, Loan Status, Current Loa..."
6,0007.cobranded,pd,80000,49,16.55,26,23,"application_key, mvar1, mvar2, mvar3, mvar4, m..."
1,0002.taiwan_creditcard,pd,30000,25,2.73,25,0,"ID, LIMIT_BAL, SEX, EDUCATION, MARRIAGE, AGE..."
3,0004.lendingclub,pd,9578,14,0.72,13,1,"credit.policy, purpose, int.rate, installment,..."
4,0005.myhom,pd,7000,10,0.27,9,1,"loan_id, age, education, proof_submitted, loan..."


Total rows across all PD datasets: 1,563,840


In [3]:
raw_lgd = raw_dataset_summary_table('lgd')
print('RAW LGD overview:')
display(raw_lgd.sort_values('rows', ascending=False))
print(f'Total rows across all LGD datasets: {int(raw_lgd["rows"].sum()):,}')


RAW LGD overview:


,dataset,task,rows,cols,size_MB,n_numeric_cols,n_object_cols,columns_preview
0,0001.heloc,lgd,67898,17,9.25,14,3,"REC, PortNum, ObsDT, DefDT, PrinBal, PayOff..."
5,0006.lgd_freddie,lgd,16002,22,2.09,13,9,"loan_id, lgd, months_since_origination, fico, ..."
6,0007.lgd_lendingclub,lgd,5627,21,0.72,12,9,"id, loan_amnt, term, int_rate, grade, emp_leng..."
1,0002.loss2,lgd,4802,72,2.73,52,20,"Alltel_Client, Loan_Category, State, UPB_At_Re..."
2,0003.axa,lgd,2545,8,0.28,8,0,"LTV, Recovery_rate, lgd_time, y_logistic, lnrr..."
3,0004.base_model,lgd,762,334,1.66,201,133,"DEAL_DocUNID, DEAL_MainID, DEAL_GoverningLawRe..."
4,0005.base_modelisation,lgd,594,282,1.49,199,83,"Ident_cliej_spm, ID_CONC_ORIGIN_CDL, CD_PROD_O..."


Total rows across all LGD datasets: 98,230


## 3. Processed dataset inventory

After `src/data/preprocessing.py` has run, each dataset lives at
`data/processed/<task>/<dataset>/` as four files (`N.npy`, `C.npy`,
`y.npy`, `info.json`). Here we report n_num / n_cat features and basic
target stats.


In [4]:
proc_pd = processed_dataset_summary_table('pd')
display(proc_pd.sort_values('rows', ascending=False))

proc_lgd = processed_dataset_summary_table('lgd')
display(proc_lgd.sort_values('rows', ascending=False))


,dataset,task,rows,n_num_features,n_cat_features,n_total_features,target_mean,target_min,target_max
5,0006,pd,532428,27,8,35,0.236327,0.0,1.0
11,0012,pd,307511,104,16,120,0.080729,0.0,1.0
2,0003,pd,233154,32,3,35,0.217071,0.0,1.0
13,0014,pd,158700,2986,0,2986,0.378091,0.0,1.0
0,0001,pd,150000,10,0,10,0.066840,0.0,1.0
10,0011,pd,105471,759,0,759,0.092755,0.0,1.0
8,0009,pd,100000,16,0,16,0.226390,0.0,1.0
6,0007,pd,80000,47,0,47,0.246213,0.0,1.0
1,0002,pd,30000,23,0,23,0.221200,0.0,1.0
3,0004,pd,9578,12,1,13,0.160054,0.0,1.0


,dataset,task,rows,n_num_features,n_cat_features,n_total_features,target_mean,target_min,target_max
0,0001,lgd,57931,8,0,8,0.704432,0.00000,1.00000
5,0006,lgd,16002,12,8,20,0.458843,0.00000,1.00000
6,0007,lgd,5627,10,7,17,0.590511,0.00000,1.00000
1,0002,lgd,4637,29,23,52,0.392032,0.00000,1.00000
2,0003,lgd,2545,2,0,2,0.228130,0.00001,0.99999
3,0004,lgd,762,114,88,202,0.323387,0.00000,1.00000
4,0005,lgd,594,179,77,256,0.330075,0.00000,1.00000


In [5]:
proc_all = pd.concat([proc_pd.assign(task='pd'), proc_lgd.assign(task='lgd')], ignore_index=True)
print('PD + LGD combined:')
display(proc_all.groupby('task')[['rows', 'n_num_features', 'n_cat_features', 'n_total_features']]
        .agg(['min', 'median', 'max', 'sum']))


PD + LGD combined:


rows                           n_num_features                     \
      min   median     max      sum            min median   max   sum   
task                                                                    
lgd   594   4637.0   57931    88098              2   12.0   179   354   
pd    999  90000.0  532428  1722026              7   19.5  2986  4052   

     n_cat_features                 n_total_features                     
                min median max  sum              min median   max   sum  
task                                                                     
lgd               0    8.0  88  203                2   20.0   256   557  
pd                0    1.0  16   46                8   21.5  2986  4098

In [6]:
plot_dataset_size_bar('pd',  out_path=FIGURES_DIR / 'pd_dataset_sizes')
plot_dataset_size_bar('lgd', out_path=FIGURES_DIR / 'lgd_dataset_sizes')
print('Saved PD + LGD dataset-size charts.')


<Figure size 1200x600 with 1 Axes>

<Figure size 1200x600 with 1 Axes>

Saved PD + LGD dataset-size charts.


## 4. PD: class balance

Default datasets are heavily imbalanced. The `imbalance_ratio` column is
``#neg / #pos`` (so 100 means a 1% positive rate; 200 means 0.5%, etc.).


In [7]:
bal = pd_target_balance_table()
display(bal)
print(f'Mean positive rate: {bal["positive_rate"].mean():.4f}')
print(f'Min  positive rate: {bal["positive_rate"].min():.4f}  (worst imbalance: {bal["imbalance_ratio"].max():.1f}:1)')
print(f'Max  positive rate: {bal["positive_rate"].max():.4f}')


,dataset,n_total,n_positive,n_negative,positive_rate,imbalance_ratio
0,0001,150000,10026,139974,0.06684,13.96
11,0012,307511,24825,282686,0.08073,11.39
10,0011,105471,9783,95688,0.09276,9.78
3,0004,9578,1533,8045,0.16005,5.25
12,0013,5960,1189,4771,0.19950,4.01
2,0003,233154,50611,182543,0.21707,3.61
1,0002,30000,6636,23364,0.22120,3.52
8,0009,100000,22639,77361,0.22639,3.42
5,0006,532428,125827,406601,0.23633,3.23
6,0007,80000,19697,60303,0.24621,3.06


Mean positive rate: 0.2207
Min  positive rate: 0.0668  (worst imbalance: 14.0:1)
Max  positive rate: 0.4000


In [8]:
plot_target_balance(out_path=FIGURES_DIR / 'pd_target_balance')
print('Saved PD class-balance chart.')


<Figure size 1200x600 with 1 Axes>

Saved PD class-balance chart.


## 5. LGD: target distribution

LGD targets live on `[0, 1]` -- the fraction of exposure that is lost
on default. Several datasets have heavy mass at exactly 0 (no loss) and
at 1 (total loss); these endpoints can cause MAPE / log-link models to
misbehave, hence our zero-exclusion bookkeeping.


In [9]:
lgd_dist = lgd_target_distribution_table()
display(lgd_dist)


,dataset,n,mean,std,median,skew,q05,q95,pct_zeros,pct_ones
0,0001,57931,0.7044,0.4170,1.0000,-0.9302,0.0000,1.0000,21.78,51.27
1,0002,4637,0.3920,0.2714,0.3515,0.5556,0.0326,0.9420,3.62,3.71
2,0003,2545,0.2281,0.3290,0.0321,1.3089,0.0000,1.0000,0.00,0.00
3,0004,762,0.3234,0.3774,0.1032,0.8270,0.0000,1.0000,10.50,11.94
4,0005,594,0.3301,0.4094,0.0435,0.7643,0.0000,1.0000,11.62,15.99
5,0006,16002,0.4588,0.3182,0.4450,0.1487,0.0000,1.0000,11.42,8.07
6,0007,5627,0.5905,0.2579,0.6334,-0.4642,0.1233,0.9335,1.48,0.32


In [10]:
plot_lgd_target_hists(out_path=FIGURES_DIR / 'lgd_target_histograms', ncols=3)
print('Saved LGD target-histogram grid.')


<Figure size 1500x1050 with 9 Axes>

Saved LGD target-histogram grid.


## 6. Missingness

Computed on **raw** data (after preprocessing, missing values have already
been imputed or replaced by sentinels). Heavily-missing columns are
candidates for dataset-specific cleanup rules in
`src/data/dataset_preprocessing.py`.


In [11]:
miss_pd = missingness_table('pd', top_k=10)
if not miss_pd.empty:
    display(miss_pd.head(40))
else:
    print('No raw missingness in PD datasets.')


,dataset,column,pct_missing
0,0001.gmsc,MonthlyIncome,19.82
1,0001.gmsc,NumberOfDependents,2.62
2,0003.vehicle_loan,Employment.Type,3.29
3,0005.myhom,education,3.50
4,0006.hackerearth,verification_status_joint,99.94
5,0006.hackerearth,desc,85.80
6,0006.hackerearth,mths_since_last_record,84.58
7,0006.hackerearth,mths_since_last_major_derog,75.02
8,0006.hackerearth,mths_since_last_delinq,51.19
9,0006.hackerearth,batch_enrolled,15.99


In [12]:
miss_lgd = missingness_table('lgd', top_k=10)
if not miss_lgd.empty:
    display(miss_lgd.head(40))
else:
    print('No raw missingness in LGD datasets.')


,dataset,column,pct_missing
0,0001.heloc,LienPos,26.81
1,0001.heloc,CurrEquifax,5.37
2,0002.loss2,Servicing_Loss,100.00
3,0002.loss2,alt_fico_code,77.91
4,0002.loss2,doc_option,60.16
5,0002.loss2,Recourse_Type,42.82
6,0002.loss2,REO_Appraisal_Date,27.55
7,0002.loss2,REO_Appraisal_Amount,27.51
8,0002.loss2,Analysis_Age,24.59
9,0002.loss2,MI_Company_Name,21.93


In [13]:
plot_missingness_heatmap('pd',  out_path=FIGURES_DIR / 'pd_missingness_heatmap')
plot_missingness_heatmap('lgd', out_path=FIGURES_DIR / 'lgd_missingness_heatmap')
print('Saved missingness heatmaps.')


<Figure size 1600x800 with 2 Axes>

<Figure size 1600x800 with 2 Axes>

Saved missingness heatmaps.


## 7. Numeric feature statistics

Per-feature descriptive stats on the processed numerical matrix. Useful
for spotting features that span many orders of magnitude (candidates for
log transforms) or have near-zero variance (candidates for removal).


In [14]:
if not proc_pd.empty:
    sample_pd = proc_pd.sort_values('rows', ascending=False).iloc[0]['dataset']
    print(f'Sample PD dataset: {sample_pd}')
    display(numeric_feature_stats('pd', sample_pd).head(20))


numeric_feature_stats(pd, 0006) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0006.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0006.parquet


Sample PD dataset: 0006


""


In [15]:
if not proc_lgd.empty:
    sample_lgd = proc_lgd.sort_values('rows', ascending=False).iloc[0]['dataset']
    print(f'Sample LGD dataset: {sample_lgd}')
    display(numeric_feature_stats('lgd', sample_lgd).head(20))


numeric_feature_stats(lgd, 0001) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\lgd\0001.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\lgd\0001.parquet


Sample LGD dataset: 0001


""


In [16]:
rows = []
for d in proc_pd['dataset']:
    stats = numeric_feature_stats('pd', d)
    if not stats.empty:
        rows.append({
            'dataset': d,
            'n_features': len(stats),
            'mean_abs_max': float(stats['max'].abs().mean()),
            'median_std': float(stats['std'].median()),
            'features_above_1e4': int((stats['max'].abs() > 1e4).sum()),
        })
display(pd.DataFrame(rows))


numeric_feature_stats(pd, 0001) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.parquet
numeric_feature_stats(pd, 0002) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.parquet
numeric_feature_stats(pd, 0003) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.parquet
numeric_feature_stats(pd, 0004) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0004.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\

""


## 8. Feature correlations

Per-dataset Pearson correlation heatmaps on the numerical block. Strong
off-diagonal blocks suggest redundancy and motivate the post-split PCA
step in `src/data/data_feeder.py`.


In [17]:
for d in proc_pd['dataset']:
    plot_correlation_heatmap('pd', d, out_path=FIGURES_DIR / f'pd_corr_{d}')
for d in proc_lgd['dataset']:
    plot_correlation_heatmap('lgd', d, out_path=FIGURES_DIR / f'lgd_corr_{d}')
print(f'Saved correlation heatmaps for {len(proc_pd) + len(proc_lgd)} datasets.')


plot_correlation_heatmap(pd, 0001) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.parquet
plot_correlation_heatmap(pd, 0002) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.parquet
plot_correlation_heatmap(pd, 0003) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.parquet
plot_correlation_heatmap(pd, 0004) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0004.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFN

Saved correlation heatmaps for 21 datasets.


## 9. Geometry: 2D PCA

Sampled 2D PCA scatter, coloured by target. For PD this gives a quick
visual on class separability; for LGD it shows the continuous target
gradient. (Datasets > 5,000 rows are sub-sampled for plotting speed.)


In [18]:
for d in proc_pd['dataset']:
    plot_pca_2d('pd', d, out_path=FIGURES_DIR / f'pd_pca_{d}', sample=5000)
for d in proc_lgd['dataset']:
    plot_pca_2d('lgd', d, out_path=FIGURES_DIR / f'lgd_pca_{d}', sample=5000)
print(f'Saved 2D PCA scatters for {len(proc_pd) + len(proc_lgd)} datasets.')


plot_pca_2d(pd, 0001) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0001.parquet
plot_pca_2d(pd, 0002) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0002.parquet
plot_pca_2d(pd, 0003) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0003.parquet
plot_pca_2d(pd, 0004) failed: Dataset file not found. Checked:
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0004.csv
  - C:\Users\U0152019\PhD Documents\Projects\1. TabPFN\TabPFNCredit\data\raw\pd\0004.parquet
plot_pca_2d(pd, 0005

Saved 2D PCA scatters for 21 datasets.


## Done

All inventory tables and figures regenerated. Inspect `figures/data_exploration/`
for the PDFs and PNGs. To re-run from scratch, simply restart the kernel
-- section 1 wipes the figures directory before the rest of the notebook runs.
